# **Note:** NAICS_CODE Based on 2017 NAICS

https://www.census.gov/naics/?input=11&year=2017

# **Read CSV to DF**

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the path to the GeoPackage file
root_path = '/content/drive/MyDrive/2025 Fall/Capstone/CAPSTONE WORK PROGRESS/Data/SafeGraph/output'

Mounted at /content/drive


In [2]:
import pandas as pd

safegraph = pd.read_csv(root_path+'/safegraph_data.csv', dtype={'NAICS_CODE': str})

print(f"Size of the data: {len(safegraph)}")
display(safegraph.head(2))

Size of the data: 298558


,BRANDS,CATEGORY_TAGS,CITY,CLOSED_ON,DOMAINS,ENCLOSED,GEOMETRY_TYPE,INCLUDES_PARKING_LOT,ISO_COUNTRY_CODE,IS_SYNTHETIC,...,POSTAL_CODE,REGION,STORE_ID,STREET_ADDRESS,SUB_CATEGORY,TOP_CATEGORY,TRACKING_CLOSED_SINCE,WEBSITE,WKT_AREA_SQ_METERS,geometry
0,[],"[""Massage""]",Gt Lon,NaN,"[""eroticlondonmassage24hours.com""]",False,POLYGON,False,GB,False,...,W1F 7NX,Greater London,NaN,Soho London,Other Personal Care Services,Personal Care Services,NaN,NaN,273.0,MULTIPOLYGON (((623171.6904224674 136052.98527...
1,[],"[""Bar or Pub""]",London,NaN,"[""thehouse.party""]",False,POLYGON,False,GB,False,...,W1F 7NU,Greater London,NaN,61 Poland Street,Drinking Places (Alcoholic Beverages),Drinking Places (Alcoholic Beverages),NaN,NaN,273.0,MULTIPOLYGON (((623171.6904224674 136052.98527...


# **Reorganize categories**

## Test data

In [3]:
# Lengths for NAICS codes
print("Confirming lengths for NAICS codes:")
print(safegraph['NAICS_CODE'].apply(lambda x: len(str(x)) if pd.notna(x) else None).value_counts().sort_index())

# Unique NAICS codes
unique_NAICS = safegraph[['NAICS_CODE']].drop_duplicates().sort_values(by='NAICS_CODE').reset_index(drop=True)
print(f"\nNumber of unique NAICS codes: {len(unique_NAICS)}")

unique_NAICS_category = safegraph[['NAICS_CODE', 'TOP_CATEGORY', 'SUB_CATEGORY']].drop_duplicates()
unique_NAICS_category = unique_NAICS_category.sort_values(by='NAICS_CODE').reset_index(drop=True)
print(f"\nNumber of unique NAICS-Category Combinations: {len(unique_NAICS_category)}")
display(unique_NAICS_category)

# Unique first two letters of NAICS
unique_first_two_naics = safegraph['NAICS_CODE'].astype(str).str[:2].unique()
unique_first_two_naics.sort()
print(f"\nUnique first two digits of NAICS_CODE (sorted):\n{unique_first_two_naics}")

Confirming lengths for NAICS codes:
NAICS_CODE
2        30
3      2174
4     30614
5      9223
6    256517
Name: count, dtype: int64

Number of unique NAICS codes: 477

Number of unique NAICS-Category Combinations: 477


,NAICS_CODE,TOP_CATEGORY,SUB_CATEGORY
0,11,"Agriculture, Forestry, Fishing and Hunting",NaN
1,111421,"Greenhouse, Nursery, and Floriculture Production",Nursery and Tree Production
2,11199,Other Crop Farming,All Other Crop Farming
3,112920,Other Animal Production,Horses and Other Equine Production
4,221114,"Electric Power Generation, Transmission and Di...",Solar Electric Power Generation
...,...,...,...
472,9231,Administration of Human Resource Programs,NaN
473,9261,Administration of Economic Programs,NaN
474,926120,Administration of Economic Programs,Regulation and Administration of Transportatio...
475,928110,National Security and International Affairs,National Security



Unique first two digits of NAICS_CODE (sorted):
['11' '22' '23' '31' '32' '33' '42' '44' '45' '48' '49' '51' '52' '53'
 '54' '55' '56' '61' '62' '71' '72' '81' '92']


## Category Level 1 based on Sector (first 2 digits of NAICS Code)

In [4]:
naics_2_digit_sector_map = {
    '11': 'Agriculture, Forestry, Fishing and Hunting',
    '21': 'Mining',
    '22': 'Utilities',
    '23': 'Construction',
    '31': 'Manufacturing',
    '32': 'Manufacturing',
    '33': 'Manufacturing',
    '42': 'Wholesale Trade',
    '44': 'Retail Trade',
    '45': 'Retail Trade',
    '48': 'Transportation and Warehousing',
    '49': 'Transportation and Warehousing',
    '51': 'Information',
    '52': 'Finance and Insurance',
    '53': 'Real Estate and Rental and Leasing',
    '54': 'Professional, Scientific, and Technical Services',
    '55': 'Management of Companies and Enterprises',
    '56': 'Administrative and Support and Waste Management and Remediation Services',
    '61': 'Educational Services',
    '62': 'Health Care and Social Assistance',
    '71': 'Arts, Entertainment, and Recreation',
    '72': 'Accommodation and Food Services',
    '81': 'Other Services (except Public Administration)',
    '92': 'Public Administration'
}

In [5]:
naics_category_map = {
    "11": "Other",
    "21": "Other",
    "22": "Service",
    "23": "Other",
    "31": "Other",
    "32": "Other",
    "33": "Other",
    "42": "Other",
    "44": "Shopping",
    "45": "Shopping",
    "48": "Transportation and Warehousing",
    "49": "Transportation and Warehousing",
    "51": "Service",
    "52": "Service",
    "53": "Service",
    "54": "Service",
    "55": "Service",
    "56": "Service",
    "61": "Education",
    "62": "Health",
    "71": "Entertainment",
    "72": "Accommodation and Food",
    "81": "Service",
    "92": "Government"
}

In [6]:
# Convert dictionaries to a DataFrame for display
naics_sector_df = pd.DataFrame(naics_2_digit_sector_map.items(), columns=['NAICS_2D', 'Sector'])
naics_sector_df['Category_Lv1'] = naics_sector_df['NAICS_2D'].map(naics_category_map)
display(naics_sector_df.sort_values(by='NAICS_2D').reset_index(drop=True))

,NAICS_2D,Sector,Category_Lv1
0,11,"Agriculture, Forestry, Fishing and Hunting",Other
1,21,Mining,Other
2,22,Utilities,Service
3,23,Construction,Other
4,31,Manufacturing,Other
5,32,Manufacturing,Other
6,33,Manufacturing,Other
7,42,Wholesale Trade,Other
8,44,Retail Trade,Shopping
9,45,Retail Trade,Shopping


In [7]:
safegraph['NAICS_2D'] = safegraph['NAICS_CODE'].astype(str).str[:2]
safegraph['Category_Lv1'] = safegraph['NAICS_2D'].map(naics_category_map)
safegraph['Sector'] = safegraph['NAICS_2D'].map(naics_2_digit_sector_map)

display(safegraph.head())

,BRANDS,CATEGORY_TAGS,CITY,CLOSED_ON,DOMAINS,ENCLOSED,GEOMETRY_TYPE,INCLUDES_PARKING_LOT,ISO_COUNTRY_CODE,IS_SYNTHETIC,...,STREET_ADDRESS,SUB_CATEGORY,TOP_CATEGORY,TRACKING_CLOSED_SINCE,WEBSITE,WKT_AREA_SQ_METERS,geometry,NAICS_2D,Category_Lv1,Sector
0,[],"[""Massage""]",Gt Lon,NaN,"[""eroticlondonmassage24hours.com""]",False,POLYGON,False,GB,False,...,Soho London,Other Personal Care Services,Personal Care Services,NaN,NaN,273.0,MULTIPOLYGON (((623171.6904224674 136052.98527...,81,Service,Other Services (except Public Administration)
1,[],"[""Bar or Pub""]",London,NaN,"[""thehouse.party""]",False,POLYGON,False,GB,False,...,61 Poland Street,Drinking Places (Alcoholic Beverages),Drinking Places (Alcoholic Beverages),NaN,NaN,273.0,MULTIPOLYGON (((623171.6904224674 136052.98527...,72,Accommodation and Food,Accommodation and Food Services
2,[],"[""Bar or Pub"",""Cocktail Lounge""]",London,NaN,"[""mlkhny.com""]",False,POLYGON,False,GB,False,...,61 Poland Street,Drinking Places (Alcoholic Beverages),Drinking Places (Alcoholic Beverages),NaN,http://www.mlkhny.com/,273.0,MULTIPOLYGON (((623171.6904224674 136052.98527...,72,Accommodation and Food,Accommodation and Food Services
3,[],"[""Accountants""]",Woodford Green,NaN,[],False,POLYGON,False,GB,False,...,82 Snakes Lane East,Offices of Certified Public Accountants,"Accounting, Tax Preparation, Bookkeeping, and ...",NaN,NaN,273.0,MULTIPOLYGON (((623171.6904224674 136052.98527...,54,Service,"Professional, Scientific, and Technical Services"
4,[],"[""Asian Food"",""Fusion Food""]",Woodford Green,NaN,"[""hungryhouse.co.uk""]",False,POLYGON,False,GB,False,...,78 Snakes Lane East,Full-Service Restaurants,Restaurants and Other Eating Places,NaN,NaN,273.0,MULTIPOLYGON (((623171.6904224674 136052.98527...,72,Accommodation and Food,Accommodation and Food Services


In [8]:
# Recategorize Transportation and Warehousing
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('48'), 'Category_Lv1'] = 'Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('49'), 'Category_Lv1'] = 'Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('493'), 'Category_Lv1'] = 'Service'

In [9]:
# Recategorize Accommodation and Food
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('72'), 'Category_Lv1'] = 'Food and Drinks'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('721'), 'Category_Lv1'] = 'Accommodation'

## Category Level 2 based on first 4 digits of NAICS Code for selected Sectors

In [10]:
# Create new col 'Category_Lv2'. Copying values from 'Sector'
safegraph['Category_Lv2'] = safegraph['Sector']

### Transportation and Warehousing

In [11]:
unique_naics_48 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('48')]['NAICS_CODE'].unique()
unique_naics_48.sort()
print("Unique NAICS_CODEs starting with '48':")
print(unique_naics_48)

Unique NAICS_CODEs starting with '48':
['481111' '4821' '483212' '484210' '484230' '485' '485119' '485210'
 '485310' '485999' '487110' '487210' '488119' '488190' '488310' '488410'
 '488510']


In [12]:
unique_naics_49 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('49')]['NAICS_CODE'].unique()
unique_naics_49.sort()
print("Unique NAICS_CODEs starting with '49':")
print(unique_naics_49)

Unique NAICS_CODEs starting with '49':
['49111' '491110' '492110' '492210' '493110' '493120']


In [13]:
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('48'), 'Category_Lv2'] = 'Uncatgorized Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('481'), 'Category_Lv2'] = 'Air Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('482'), 'Category_Lv2'] = 'Rail Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('483'), 'Category_Lv2'] = 'Uncatgorized Water Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4831'), 'Category_Lv2'] = 'Deep Sea, Coastal, and Great Lakes Water Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4832'), 'Category_Lv2'] = 'Inland Water Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('484'), 'Category_Lv2'] = 'Truck Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('485'), 'Category_Lv2'] = 'Uncatgorized Transit and Ground Passenger Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4851'), 'Category_Lv2'] = 'Urban Transit Systems'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4852'), 'Category_Lv2'] = 'Interurban and Rural Bus Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4853'), 'Category_Lv2'] = 'Taxi and Limousine Service'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4854'), 'Category_Lv2'] = 'School and Employee Bus Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4855'), 'Category_Lv2'] = 'Charter Bus Industry'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4859'), 'Category_Lv2'] = 'Other Transit and Ground Passenger Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('486'), 'Category_Lv2'] = 'Pipeline Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('487'), 'Category_Lv2'] = 'Uncatgorized Scenic and Sightseeing Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4871'), 'Category_Lv2'] = 'Scenic and Sightseeing Transportation, Land'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4872'), 'Category_Lv2'] = 'Scenic and Sightseeing Transportation, Water'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4879'), 'Category_Lv2'] = 'Scenic and Sightseeing Transportation, Other'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('488'), 'Category_Lv2'] = 'Support Activities for Transportation'

safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('49'), 'Category_Lv2'] = 'Uncatgorized Transportation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('491'), 'Category_Lv2'] = 'Postal Service'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('492'), 'Category_Lv2'] = 'Uncatgorized Couriers and Messengers'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4921'), 'Category_Lv2'] = 'Couriers and Express Delivery Services'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('4922'), 'Category_Lv2'] = 'Local Messengers and Local Delivery'

safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('493'), 'Category_Lv2'] = 'Warehousing and Storage'

### Accommodation and Food

In [14]:
unique_naics_72 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('72')]['NAICS_CODE'].unique()
unique_naics_72.sort()
print("Unique NAICS_CODEs starting with '72':")
print(unique_naics_72)

Unique NAICS_CODEs starting with '72':
['721110' '721120' '721199' '7212' '721211' '721214' '722' '722320'
 '722330' '722410' '7225' '72251' '722511' '722513' '722515']


In [15]:
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('72'), 'Category_Lv2'] = 'Uncatgorized Accommodation and Food Services'

safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('721'), 'Category_Lv2'] = 'Uncatgorized Accommodation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7211'), 'Category_Lv2'] = 'Traveler Accommodation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7212'), 'Category_Lv2'] = 'RV (Recreational Vehicle) Parks and Recreational Camps'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7213'), 'Category_Lv2'] = 'Rooming and Boarding Houses, Dormitories, and Workers\' Camps'

safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('722'), 'Category_Lv2'] = 'Uncatgorized Food Services and Drinking Places'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7223'), 'Category_Lv2'] = 'Special Food Services'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7224'), 'Category_Lv2'] = 'Drinking Places (Alcoholic Beverages)'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7225'), 'Category_Lv2'] = 'Uncatgorized Restaurants and Other Eating Places'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('722511'), 'Category_Lv2'] = 'Full-Service Restaurants'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('722513'), 'Category_Lv2'] = 'Limited-Service Restaurants'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('722514'), 'Category_Lv2'] = 'Cafeterias, Grill Buffets, and Buffets'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('722515'), 'Category_Lv2'] = 'Snack and Nonalcoholic Beverage Bars'

### Shopping

In [16]:
unique_naics_44 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('44')]['NAICS_CODE'].unique()
unique_naics_44.sort()
print("Unique NAICS_CODEs starting with '44':")
print(unique_naics_44)

Unique NAICS_CODEs starting with '44':
['441110' '441120' '441210' '441222' '441228' '441310' '441320' '442110'
 '4422' '442210' '442291' '442299' '443141' '443142' '444110' '444120'
 '444130' '444190' '444210' '444220' '445110' '445120' '445210' '445220'
 '445230' '445291' '445292' '445299' '445310' '446110' '446120' '446130'
 '446191' '446199' '4471' '447110' '4481' '448110' '448120' '448130'
 '44814' '448140' '448150' '448190' '448210' '44831' '448310' '448320']


In [17]:
unique_naics_45 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('45')]['NAICS_CODE'].unique()
unique_naics_45.sort()
print("Unique NAICS_CODEs starting with '45':")
print(unique_naics_45)

Unique NAICS_CODEs starting with '45':
['451110' '451120' '451130' '451140' '451211' '451212' '452210' '4523'
 '45231' '452311' '452319' '453110' '453210' '453220' '453310' '4539'
 '453910' '453920' '45399' '453991' '453998' '4541' '454110' '454310'
 '45439' '454390']


In [18]:
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('44'), 'Category_Lv2'] = 'Uncatgorized Retail Trade'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('441'), 'Category_Lv2'] = 'Motor Vehicle and Parts Dealers'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('442'), 'Category_Lv2'] = 'Furniture and Home Furnishings Stores'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('443'), 'Category_Lv2'] = 'Electronics and Appliance Stores'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('444'), 'Category_Lv2'] = 'Building Material and Garden Equipment and Supplies Dealers'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('445'), 'Category_Lv2'] = 'Food and Beverage Stores'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('446'), 'Category_Lv2'] = 'Health and Personal Care Stores'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('447'), 'Category_Lv2'] = 'Gasoline Stations'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('448'), 'Category_Lv2'] = 'Clothing and Clothing Accessories Stores'

In [19]:
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('45'), 'Category_Lv2'] = 'Uncatgorized Retail Trade'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('451'), 'Category_Lv2'] = 'Sporting Goods, Hobby, Musical Instrument, and Book Stores'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('452'), 'Category_Lv2'] = 'General Merchandise Stores'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('453'), 'Category_Lv2'] = 'Miscellaneous Store Retailers'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('454'), 'Category_Lv2'] = 'Nonstore Retailers'

### Health

In [20]:
unique_naics_62 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('62')]['NAICS_CODE'].unique()
unique_naics_62.sort()
print("Unique NAICS_CODEs starting with '62':")
print(unique_naics_62)

Unique NAICS_CODEs starting with '62':
['621111' '621112' '621210' '621310' '621320' '621330' '621340' '621399'
 '62149' '621492' '621498' '621512' '621610' '621910' '621999' '622110'
 '622210' '622310' '623110' '623311' '623312' '624110' '624120' '624190'
 '62422' '624221' '624410']


In [21]:
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('62'), 'Category_Lv2'] = 'Uncatgorized Health Care and Social Assistance'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('621'), 'Category_Lv2'] = 'Ambulatory Health Care Services'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('622'), 'Category_Lv2'] = 'Hospitals'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('623'), 'Category_Lv2'] = 'Nursing and Residential Care Facilities'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('624'), 'Category_Lv2'] = 'Social Assistance'

### Entertainment

In [22]:
unique_naics_71 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('71')]['NAICS_CODE'].unique()
unique_naics_71.sort()
print("Unique NAICS_CODEs starting with '71':")
print(unique_naics_71)

Unique NAICS_CODEs starting with '71':
['7111' '71111' '711110' '711120' '71113' '711130' '71119' '711190' '7112'
 '711211' '711212' '711219' '7113' '711310' '711320' '7114' '711410'
 '7115' '71151' '711510' '7121' '712110' '712120' '712130' '712190' '7131'
 '713110' '713120' '713210' '713290' '7139' '713910' '713920' '713930'
 '713940' '713950' '713990']


In [23]:
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('71'), 'Category_Lv2'] = 'Uncatgorized Arts, Entertainment, and Recreation'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('711'), 'Category_Lv2'] = 'Performing Arts, Spectator Sports, and Related Industries'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('712'), 'Category_Lv2'] = 'Museums, Historical Sites, and Similar Institutions'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('713'), 'Category_Lv2'] = 'Uncatgorized Amusement, Gambling, and Recreation Industries'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7131'), 'Category_Lv2'] = 'Amusement Parks and Arcades'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7132'), 'Category_Lv2'] = 'Gambling Industries'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('7139'), 'Category_Lv2'] = 'Uncatgorized Other Amusement and Recreation Industries'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('71391'), 'Category_Lv2'] = 'Golf Courses and Country Clubs'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('71392'), 'Category_Lv2'] = 'Skiing Facilities'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('71393'), 'Category_Lv2'] = 'Marinas'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('71394'), 'Category_Lv2'] = 'Fitness and Recreational Sports Centers'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('71395'), 'Category_Lv2'] = 'Bowling Centers'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('71399'), 'Category_Lv2'] = 'All Other Amusement and Recreation Industries'

### Education

In [24]:
unique_naics_61 = safegraph[safegraph['NAICS_CODE'].astype(str).str.startswith('61')]['NAICS_CODE'].unique()
unique_naics_61.sort()
print("Unique NAICS_CODEs starting with '61':")
print(unique_naics_61)

Unique NAICS_CODEs starting with '61':
['611110' '611210' '611310' '611420' '61143' '611430' '61151' '611511'
 '611519' '611610' '611620' '611630' '611691' '611692' '611699' '611710']


In [25]:
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('61'), 'Category_Lv2'] = 'Uncatgorized Educational Services'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('6111'), 'Category_Lv2'] = 'Elementary and Secondary Schools'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('6112'), 'Category_Lv2'] = 'Junior Colleges'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('6113'), 'Category_Lv2'] = 'Colleges, Universities, and Professional Schools'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('6114'), 'Category_Lv2'] = 'Business Schools and Computer and Management Training'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('6115'), 'Category_Lv2'] = 'Technical and Trade Schools'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('6116'), 'Category_Lv2'] = 'Uncatgorized Other Schools and Instruction'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('61161'), 'Category_Lv2'] = 'Fine Arts Schools'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('61162'), 'Category_Lv2'] = 'Sports and Recreation Instruction'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('61163'), 'Category_Lv2'] = 'Language Schools'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('61169'), 'Category_Lv2'] = 'Uncatgorized All Other Schools and Instruction'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('611691'), 'Category_Lv2'] = 'Exam Preparation and Tutoring'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('611692'), 'Category_Lv2'] = 'Automobile Driving Schools'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('611699'), 'Category_Lv2'] = 'All Other Miscellaneous Schools and Instruction'
safegraph.loc[safegraph['NAICS_CODE'].astype(str).str.startswith('6117'), 'Category_Lv2'] = 'Educational Support Services'

## NAICS Code to Category Mapping Table

In [30]:
# Extract unique combinations of NAICS_CODE, Category_Lv1, and Category_Lv2
naics_mapping = safegraph[['NAICS_CODE', 'Category_Lv1', 'Category_Lv2']].drop_duplicates()

# Sort by NAICS_CODE for better readability
naics_mapping = naics_mapping.sort_values(by='NAICS_CODE').reset_index(drop=True)

print(f"Total unique NAICS to Category mappings: {len(naics_mapping)}")
display(naics_mapping)

Total unique NAICS to Category mappings: 477


,NAICS_CODE,Category_Lv1,Category_Lv2
0,11,Other,"Agriculture, Forestry, Fishing and Hunting"
1,111421,Other,"Agriculture, Forestry, Fishing and Hunting"
2,11199,Other,"Agriculture, Forestry, Fishing and Hunting"
3,112920,Other,"Agriculture, Forestry, Fishing and Hunting"
4,221114,Service,Utilities
...,...,...,...
472,9231,Government,Public Administration
473,9261,Government,Public Administration
474,926120,Government,Public Administration
475,928110,Government,Public Administration


# **Export to CSV**

In [26]:
safegraph.to_csv(root_path+'/safegraph_recategorized.csv', index=False)
print(f"DataFrame saved to {root_path}/safegraph_recategorized.csv")

DataFrame saved to /content/drive/MyDrive/2025 Fall/Capstone/CAPSTONE WORK PROGRESS/Data/SafeGraph/output/safegraph_recategorized.csv


# **Plot Treemap**

## Group Data

In [27]:
# Group by 'Category_Lv1' and 'Category_Lv2' and count occurrences
treemap_nested_data = safegraph.groupby(['Category_Lv1', 'Category_Lv2']).size().reset_index(name='Count')
display(treemap_nested_data)

,Category_Lv1,Category_Lv2,Count
0,Accommodation,RV (Recreational Vehicle) Parks and Recreation...,60
1,Accommodation,Traveler Accommodation,3624
2,Education,All Other Miscellaneous Schools and Instruction,2467
3,Education,Automobile Driving Schools,930
4,Education,Business Schools and Computer and Management T...,21
...,...,...,...
72,Transportation,Support Activities for Transportation,373
73,Transportation,Taxi and Limousine Service,3273
74,Transportation,Truck Transportation,2508
75,Transportation,Uncatgorized Transit and Ground Passenger Tran...,622


## go.Treemap

In [28]:
import plotly.graph_objects as go
import plotly.colors
from plotly.subplots import make_subplots

# Function to darken a hex color
def darken_hex_color(hex_color, factor=0.8):
    hex_color = hex_color.lstrip('#')
    rgb = tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))
    darker_rgb = tuple(max(0, int(c * factor)) for c in rgb)
    return '#%02x%02x%02x' % darker_rgb

total_safegraph_count = len(safegraph)

# --- Data for Treemap 1 (Category Level 1) ---
category_lv1_dist = safegraph['Category_Lv1'].value_counts().reset_index()
category_lv1_dist.columns = ['Category_Lv1', 'Count']

labels_t1 = category_lv1_dist['Category_Lv1'].tolist()
parents_t1 = ["" for _ in labels_t1]
values_t1 = category_lv1_dist['Count'].tolist()

# Calculate percentages for T1 and create custom text
percentages_t1 = [round((count / total_safegraph_count) * 100, 2) for count in category_lv1_dist['Count']]
text_t1 = [f"{labels_t1[i]}<br>{values_t1[i]}<br>{percentages_t1[i]:.2f}%" for i in range(len(labels_t1))]

# --- Data for Treemap 2 (Category Level 2 Nested) ---
# Group by 'Category_Lv1' and 'Category_Lv2' and count occurrences
treemap_nested_data = safegraph.groupby(['Category_Lv1', 'Category_Lv2']).size().reset_index(name='Count')

# Calculate total counts for Level 1 categories to use for parent nodes
level1_counts_t2 = treemap_nested_data.groupby('Category_Lv1')['Count'].sum().reset_index()
level1_counts_t2.columns = ['Category_Lv1', 'Total_Count']

labels_t2 = []
parents_t2 = []
values_t2 = []
text_t2 = [] # List to hold custom text for T2 nodes

# --- Create a single, consistent color map for Level 1 categories across both plots ---
# Get all unique Level 1 categories from both data sources to ensure consistent mapping
all_unique_level1_categories = pd.concat([category_lv1_dist['Category_Lv1'], level1_counts_t2['Category_Lv1']]).unique()
all_unique_level1_categories.sort() # Ensure consistent order for color assignment

color_palette = plotly.colors.qualitative.Plotly # Using a qualitative palette for distinct colors
global_level1_color_map = {cat: color_palette[i % len(color_palette)] for i, cat in enumerate(all_unique_level1_categories)}

# Assign colors for Treemap 1 using the global map
color_codes_t1 = [global_level1_color_map[label] for label in labels_t1]

# --- Build data and colors for Treemap 2 using the global map ---
color_codes_t2 = []

# Add Level 1 categories to the treemap data (for T2)
for _, row in level1_counts_t2.iterrows():
    labels_t2.append(row['Category_Lv1'])
    parents_t2.append("") # Top level has an empty parent
    values_t2.append(row['Total_Count'])
    color_codes_t2.append(global_level1_color_map[row['Category_Lv1']]) # Use global map

    # Calculate percentage for Level 1 parents relative to overall total
    percentage_overall = round((row['Total_Count'] / total_safegraph_count) * 100, 2)
    text_t2.append(f"{row['Category_Lv1']}<br>{row['Total_Count']}<br>{percentage_overall:.2f}%")

# Add Level 2 categories to the treemap data (for T2)
for _, row in treemap_nested_data.iterrows():
    labels_t2.append(row['Category_Lv2'])
    parents_t2.append(row['Category_Lv1']) # Parent is the Level 1 category
    parent_color = global_level1_color_map[row['Category_Lv1']] # Use global map
    darker_color = darken_hex_color(parent_color, factor=0.9) # Darken the parent color for the child
    values_t2.append(row['Count'])
    color_codes_t2.append(darker_color)

    # Calculate percentage for Level 2 children relative to the total count
    percentage_of_total = round((row['Count'] / total_safegraph_count) * 100, 2)
    text_t2.append(f"{row['Category_Lv2']}<br>{row['Count']}<br>{percentage_of_total:.2f}%")

# --- Create Subplots ---
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type':'domain'}, {'type':'domain'}]],
    # subplot_titles=('Category Level 1', 'Category Level 2')
)

# Add Treemap 1 to the first subplot
fig.add_trace(go.Treemap(
    labels=labels_t1,
    parents=parents_t1,
    values=values_t1,
    marker=dict(colors=color_codes_t1, cornerradius=4),
    name='Level 1 Categories',
    text=text_t1,
    textinfo='text', # Use custom text
    hoverinfo='text' # Set hoverinfo to display only custom text
), row=1, col=1)

# Add Treemap 2 to the second subplot
fig.add_trace(go.Treemap(
    labels=labels_t2,
    parents=parents_t2,
    values=values_t2,
    marker=dict(colors=color_codes_t2, cornerradius=4),
    branchvalues="total", # Show parent value as sum of children
    name='Nested Categories',
    text=text_t2,
    textinfo='text', # Use custom text
    hoverinfo='text' # Set hoverinfo to display only custom text
), row=1, col=2)

fig.update_layout(
    height=600, # Adjust height as needed
    margin = dict(t=25, l=10, r=10, b=10)
)

fig.show()

In [29]:
# To export the plot to an interactive HTML file:
# This allows you to view the interactive plot in a web browser.
fig.write_html(root_path+"/poi_category_treemaps.html")
print("Plot exported to poi_category_treemaps.html")

Plot exported to poi_category_treemaps.html
